In [3]:
import joblib
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, classification_report

X_train_processed = joblib.load('X_train_processed.pkl')
X_test_processed = joblib.load('X_test_processed.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')

scale = (y_train == 0).sum() / (y_train == 1).sum()

best_model = LGBMClassifier(
    n_estimators=173,
    learning_rate=0.010634,
    num_leaves=37,
    min_child_samples=93,
    scale_pos_weight=scale,
    random_state=42,
    verbose=-1
)

best_model.fit(X_train_processed, y_train)

train_proba = best_model.predict_proba(X_train_processed)[:, 1]
test_proba = best_model.predict_proba(X_test_processed)[:, 1]
y_pred_train = best_model.predict(X_train_processed)
y_pred_test = best_model.predict(X_test_processed)

print('Train PR-AUC:', average_precision_score(y_train, train_proba))
print('Test PR-AUC:', average_precision_score(y_test, test_proba))
print('\nTrain Classification Report:')
print(classification_report(y_train, y_pred_train))
print('\nTest Classification Report:')
print(classification_report(y_test, y_pred_test))

joblib.dump(best_model, 'best_model_lgb.pkl')
print('Model kaydedildi!')

Train PR-AUC: 0.7890430536517761
Test PR-AUC: 0.6527088560280899

Train Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.84      0.90       602
           1       0.65      0.89      0.75       198

    accuracy                           0.86       800
   macro avg       0.81      0.87      0.83       800
weighted avg       0.88      0.86      0.86       800


Test Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.84      0.89       151
           1       0.64      0.86      0.73        49

    accuracy                           0.84       200
   macro avg       0.79      0.85      0.81       200
weighted avg       0.87      0.84      0.85       200

Model kaydedildi!


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [1]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, classification_report
import joblib

preprocessor = joblib.load('preprocessor.pkl')
X_train_processed = joblib.load('X_train_processed.pkl')
X_test_processed = joblib.load('X_test_processed.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')
scale = (y_train == 0).sum() / (y_train == 1).sum()


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 250),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
        'num_leaves': trial.suggest_int('num_leaves', 15, 40),
        'min_child_samples': trial.suggest_int('min_child_samples', 80, 150),
        'scale_pos_weight': scale,
        'random_state': 42
    }

    model = LGBMClassifier(**params, verbose=-1)
    cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_processed,  y_train, cv=cv, scoring='average_precision')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=150)

print('Best PR-AUC:', study.best_value)
print('Best params:', study.best_params)


best_model = LGBMClassifier(**study.best_params, scale_pos_weight=scale, random_state=42, verbose=-1)
best_model.fit(X_train_processed, y_train)

train_proba = best_model.predict_proba(X_train_processed)[:, 1]
test_proba = best_model.predict_proba(X_test_processed)[:, 1]

y_pred_train = best_model.predict(X_train_processed)
y_pred_test = best_model.predict(X_test_processed)

print('Train PR-AUC:', average_precision_score(y_train, train_proba))
print('Test PR-AUC:', average_precision_score(y_test, test_proba))
print('\nTest Classification Report:')
print('Train Classification Report:')
print(classification_report(y_train, y_pred_train))
print('Test Classification Report:')
print(classification_report(y_test, y_pred_test))


joblib.dump(best_model, 'best_model_lgb.pkl')
print('Model kaydedildi!')


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-30 10:40:06,671] A new study created in memory with name: no-name-0373ea6c-0c28-4c8f-863b-a2363962f1c9
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMC

Best PR-AUC: 0.7297941513010644
Best params: {'n_estimators': 118, 'learning_rate': 0.014985822058609093, 'num_leaves': 33, 'min_child_samples': 86}
Train PR-AUC: 0.7846648830596084
Test PR-AUC: 0.6111337956245986

Test Classification Report:
Train Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.84      0.90       602
           1       0.65      0.89      0.75       198

    accuracy                           0.86       800
   macro avg       0.81      0.87      0.83       800
weighted avg       0.88      0.86      0.86       800

Test Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.84      0.89       151
           1       0.64      0.86      0.73        49

    accuracy                           0.84       200
   macro avg       0.79      0.85      0.81       200
weighted avg       0.87      0.84      0.85       200

Model kaydedildi!


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib

preprocessor = joblib.load('preprocessor.pkl')
X_train_processed = joblib.load('X_train_processed.pkl')
X_test_processed = joblib.load('X_test_processed.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')
scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline = Pipeline(steps=[
    ('model', LGBMClassifier(n_estimators=100, scale_pos_weight=scale, random_state=42))
])

pipeline.fit(X_train_processed, y_train)

y_pred_proba = pipeline.predict_proba(X_test_processed)[:, 1]

pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"PR-AUC: {pr_auc:.4f}")
y_pred = pipeline.predict(X_test_processed)
print(classification_report(y_test, y_pred))
print(f"scale_pos_weight: {scale:.2f}")




PR-AUC: 0.5905
              precision    recall  f1-score   support

           0       0.87      0.86      0.87       151
           1       0.59      0.61      0.60        49

    accuracy                           0.80       200
   macro avg       0.73      0.74      0.73       200
weighted avg       0.80      0.80      0.80       200

scale_pos_weight: 3.04


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [3]:
print(preprocessor.get_feature_names_out())

['cat__policy_state' 'cat__policy_csl' 'cat__insured_sex'
 'cat__insured_education_level' 'cat__insured_occupation'
 'cat__insured_hobbies' 'cat__insured_relationship' 'cat__incident_type'
 'cat__collision_type' 'cat__incident_severity'
 'cat__authorities_contacted' 'cat__incident_state' 'cat__incident_city'
 'cat__property_damage' 'cat__police_report_available' 'cat__auto_make'
 'remainder__months_as_customer' 'remainder__age'
 'remainder__policy_deductable' 'remainder__policy_annual_premium'
 'remainder__umbrella_limit' 'remainder__capital-gains'
 'remainder__capital-loss' 'remainder__incident_hour_of_the_day'
 'remainder__number_of_vehicles_involved' 'remainder__bodily_injuries'
 'remainder__witnesses' 'remainder__total_claim_amount'
 'remainder__injury_claim' 'remainder__property_claim'
 'remainder__vehicle_claim' 'remainder__auto_year'
 'remainder__incident_year' 'remainder__policy_bind_year'
 'remainder__injury_ratio']
